# 0 - Test API OECD

## Importation des modules

In [1]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import pandas as pd
import json
from io import StringIO
import sys
import yaml
from typing import Dict, List
from lxml import etree
from xml.etree import ElementTree as ET
# Module de scrapping
import requests

# Ajout du chemin
sys.path.append('..')

# Modules du package
from macroforecast.datasets.client import APIClient
from macroforecast.datasets.sdmx import SDMXURLBuilder
from macroforecast.datasets.oecd import OECDClient
from macroforecast.datasets.structures import DataflowStructureRegistry

# Chargement de la configuration
with open('../config/datasets/oecd.yaml') as file:
    config = yaml.safe_load(file)

# Chargement du fichier de paramètres
with open('../parameters/oecd.json') as file:
    oecd_parameters = json.load(file)

## Extraction de l'ensemble des dataflows

In [2]:
# Initialisation de l'url
url = "https://sdmx.oecd.org/public/rest/dataflow/all"

# Exécution de la requête
response = requests.get(url, headers={'Accept': 'application/xml'})

# Parsing du XML
root = ET.fromstring(response.content)

# Namespaces SDMX (à adapter selon la structure)
namespaces = {
    'mes': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message',
    'str': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure',
    'com': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common'
}

# Extraction des dataflows
dataflows = []
for df in root.findall('.//str:Dataflow', namespaces):
    dataflow_id = df.get('id')
    agency_id = df.get('agencyID')
    version = df.get('version')
    
    # Extraction du nom
    name_elem = df.find('.//com:Name', namespaces)
    name = name_elem.text if name_elem is not None else None
    
    dataflows.append({
        'id': dataflow_id,
        'agencyID': agency_id,
        'version': version,
        'name': name
    })

# Conversion en DataFrame
df = pd.DataFrame(dataflows)

#df.to_csv("dataflows.csv")

df.head()

,id,agencyID,version,name
0,SEEA_AEA_A,ESTAT,1.4,Air Emissions Accounts
1,DF_SDG_GLC,IAEG-SDGs,1.20,SDG Country Global Dataflow
2,DF_SDG_GLH,IAEG-SDGs,1.20,SDG Harmonized Global Dataflow
3,DSD_FUA_CLIM@DF_CLIM_PROJ,OECD.CFE.EDS,1.4,"Climate projections by scenario, 2030–2060 – C..."
4,DSD_FUA_CLIM@DF_COASTAL_FLOOD,OECD.CFE.EDS,1.1,Coastal flooding - Cities and FUAs


## Fonctions utilitaires

In [3]:


def get_dataflow_structure(agency: str, dataflow_id: str, version: str = "latest") -> Dict:
    """
    Récupère la structure complète d'un dataflow SDMX de l'OCDE.
    
    Args:
        agency: L'agence (ex: 'OECD.SDD.STES')
        dataflow_id: L'identifiant du dataflow (ex: 'DSD_KEI')
        version: La version (défaut: 'latest')
    
    Returns:
        Dictionnaire contenant les dimensions et leurs codelists
    """
    # Récupérer la DSD
    dsd_url = f"https://sdmx.oecd.org/public/rest/v2/structure/datastructure/{agency}/{dataflow_id}/{version}"
    response = requests.get(dsd_url, headers={"Accept": "application/xml"})
    response.raise_for_status()
    
    root = etree.fromstring(response.content)
    
    namespaces = {
        'mes': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message',
        'str': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure',
        'com': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common'
    }
    
    structure = {
        'dimensions': [],
        'attributes': [],
        'measures': []
    }
    
    # Extraire les dimensions
    dimensions = root.xpath('//str:Dimension', namespaces=namespaces)
    for dim in sorted(dimensions, key=lambda x: int(x.get('position', 0))):
        dim_id = dim.get('id')
        position = dim.get('position')
        
        # Récupérer la référence à la codelist
        codelist_ref = dim.xpath('.//str:Enumeration/Ref', namespaces=namespaces)
        codelist_id = codelist_ref[0].get('id') if codelist_ref else None
        
        structure['dimensions'].append({
            'id': dim_id,
            'position': int(position) if position else None,
            'codelist': codelist_id
        })
    
    return structure

def get_codelist(agency: str, codelist_id: str) -> Dict[str, Dict[str, str]]:
    """
    Récupère les codes d'une codelist.
    
    Args:
        agency: L'agence (ex: 'OECD.SDD.STES')
        codelist_id: L'identifiant de la codelist
    
    Returns:
        Dictionnaire {code_id: {'en': label_en, 'fr': label_fr}}
    """
    url = f"https://sdmx.oecd.org/public/rest/v2/structure/codelist/{agency}/{codelist_id}"
    response = requests.get(url, headers={"Accept": "application/xml"})
    response.raise_for_status()
    
    root = etree.fromstring(response.content)
    
    namespaces = {
        'str': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure',
        'com': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common'
    }
    
    codes = {}
    for code in root.xpath('//str:Code', namespaces=namespaces):
        code_id = code.get('id')
        name_fr = code.xpath('.//com:Name[@xml:lang="fr"]/text()', namespaces=namespaces)
        name_en = code.xpath('.//com:Name[@xml:lang="en"]/text()', namespaces=namespaces)
        
        codes[code_id] = {
            'en': name_en[0] if name_en else '',
            'fr': name_fr[0] if name_fr else ''
        }
    
    return codes

## Test des requêtes

### Key economic indicators

In [4]:
# Extraction de la structure depuis l'API
# Initialisation du client
oecd_client = OECDClient()
# Extraction de la structure
structure = oecd_client.get_structure(agency="OECD.SDD.STES", dataflow="DSD_KEI@DF_KEI", version="4.0")
structure.to_dict()

{'agency': 'OECD.SDD.STES',
 'dataflow': 'DSD_KEI@DF_KEI',
 'num_dimensions': 7,
 'dimensions': [{'name': 'REF_AREA', 'position': 0},
  {'name': 'FREQ', 'position': 1},
  {'name': 'MEASURE', 'position': 2},
  {'name': 'UNIT_MEASURE', 'position': 3},
  {'name': 'ACTIVITY', 'position': 4},
  {'name': 'ADJUSTMENT', 'position': 5},
  {'name': 'TRANSFORMATION', 'position': 6}],
 'description': None}

In [8]:
# Initialisation de la structure des dataflows à partir 
dataflow_structures = DataflowStructureRegistry(config_path='../parameters/oecd.json')

# Initialisation du client
oecd_client = OECDClient(structure_registry=dataflow_structures)

# Requête des données
df = oecd_client.get_data(
    agency= "OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "MEASURE": ["LI"],
        "REF_AREA": ["FRA", "DEU"],
        "FREQ": ["M"]
    },
    start_period=None,
    end_period=None,
    last_n_observations=None
)

df.head()

HTTP error: 404 Client Error: Not Found for url: https://sdmx.oecd.org/public/rest/v2/data/dataflow/OECD.SDD.STES/DSD_KEI@DF_KEI/+/FRA+DEU.M.LI.*.*.*.*?dimensionAtObservation=AllDimensions&format=csvfilewithlabels
Response content: NoRecordsFound


HTTPError: 404 Client Error: Not Found for url: https://sdmx.oecd.org/public/rest/v2/data/dataflow/OECD.SDD.STES/DSD_KEI@DF_KEI/+/FRA+DEU.M.LI.*.*.*.*?dimensionAtObservation=AllDimensions&format=csvfilewithlabels

### Main economic indicators - cyclical indicators

### Monthly financial indicators

### Business Tendency and Consumer Opinion Surveys

### Producer Price Indices

In [ ]:
### Main Economic Indicators (MEI) - National Accounts

In [ ]:
### Consumer Price Index

In [ ]:
### Analytical House Price Indicators